# Note decoding

Nov 1 and 2. How accurately do notes come out of clean audio, and which settings help?

**Data:** GuitarSet (CC BY 4.0), and 20 public-domain Mutopia pieces rendered with the Salamander piano (CC BY 3.0). **Measure:** note F1 at 50 ms and 50 cents against labels.

**What it decided:** guitar keeps Basic Pitch's 127.7 ms minimum note (0.732 to 0.777 on unseen players). Piano uses a stricter onset threshold and no melodia trick (in a band 0.684 to 0.757 on unseen pieces). Both were run again on Nov 2; see `results/`.

This is a selection of the cells from that work, with the output they printed. They rely on earlier cells and on a Drive folder of datasets and outputs that is not in this repository.


## Guitar: labelled cases and the current decoder


In [ ]:
"""D1a: write today's plan beside the runs, then see what GuitarSet actually publishes.

GuitarSet is the only data within reach that carries real note annotations, and it is solo guitar,
so the decoder can be measured without separation confounding it.
"""

import urllib.request

PLAN = """# Measurement plan — 2026-11-01

Everything is measured on this Colab VM. Implementation lands in the repo at the end of the day.

## Frozen metrics

| id | metric | definition |
|---|---|---|
| S1 | separation | SI-SDR improvement over the raw mixture, dB, mono, 48 kHz, no time alignment |
| N1 | separation seen through notes | onset+pitch F1, 50 ms / 50 cents, offset ignored, against the decoder's own output on the clean stem |
| N2 | octave-blind notes | N1 with pitch classes, separating octave errors from pitch errors |
| N3 | decoder accuracy | the same F1 against GuitarSet's published note annotations, which are real labels |
| P1 | latency | wall-clock seconds per isolation, model load reported separately |
| P2 | memory | peak VRAM, GiB |
| P3 | cost | GPU seconds times the Modal rate, dollars per isolation |

N1 and N3 are orthogonal: N1 asks how close separation gets to the ideal-stem result, N3 asks how
accurate the decoder is on audio that needs no separation. N1 cannot judge a decoder change,
because its reference is the decoder's own output.

## Decision rules

1. Demucs stays the default for its six categories if its N1 beats SAM by at least 0.05. **Met**:
   0.532 against 0.388 over 16 targets.
2. SAM keeps prompts outside those six only if its N1 beats the raw mixture by at least 0.05.
   **Failed**: 0.007 against 0.036 over 4 targets. Those prompts ship without note claims.
3. The cross-check selector ships only if it adds at least 0.05 mean N1 over Demucs alone and makes
   no case worse. **Failed**: +0.005, and five cases worse.
4. Draws drop to one unless a runtime selector earns 0.05. **Applied**, and Judge was removed.
5. A decoder change ships only if it adds at least 0.03 mean N3 on clean audio and costs no
   instrument class more than 0.01 N1 with the reference recomputed by the same decoder. Changes
   already measured as negative are not retried.

## Where quality stands

Shipped path, 15 valid cases: **56.7%** of the clean-stem result.

| class | n | N1 | range |
|---|---:|---:|---|
| bass | 2 | 74.6% | 72.8 - 76.5 |
| vocals | 2 | 69.0% | 66.4 - 71.5 |
| guitar | 8 | 63.6% | 35.7 - 85.1 |
| piano | 3 | 18.4% | 2.3 - 46.0 |
| outside the six categories | 4 | 0.7% | raw mixture 3.6% |

Baselines: raw mixture 27.1%, the SAM path 38.8%.

## Steps

| step | work | status |
|---|---|---|
| M1 | holdout-004: 10 targets from the 5 songs no earlier run touched | done |
| M2 | N1 and N2 for every case and path | done |
| M3 | cross-check selector over 20 cases | done, rejected |
| M4 | engine routing, one draw, no ranker | written, checks pass |
| M5 | P1, P2, P3 for the shipped configuration | done |
| D1 | fetch GuitarSet audio and annotations here | running |
| D2 | build 12 solo cases, 30 s at 48 kHz, labels from the published annotations | |
| D3 | N3 for the production decoder: the ceiling clean audio allows today | |
| D4 | predeclared one-factor sweep: melodia_trick off; minimum note length 80 ms against 30 and 127.7; onset threshold 0.6 against 0.5. Activations cached once and re-decoded | |
"""
(BASE / "runs" / "PLAN-2026-11-01.md").write_text(PLAN)
print(f"plan written to {BASE / 'runs' / 'PLAN-2026-11-01.md'}\n")

ZENODO_RECORD = "3371780"
with urllib.request.urlopen(
    f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=60
) as response:
    record = json.load(response)
print(f"{record['metadata']['title']}")
print(f"license: {record['metadata'].get('license', {}).get('id', '?')}\n")
for item in sorted(record["files"], key=lambda f: -f["size"]):
    print(f"  {item['size'] / 2**20:9.1f} MiB  {item['key']}")


plan written to /content/drive/MyDrive/mixray-quality/runs/PLAN-2026-11-01.md

GuitarSet
license: cc-by-4.0

     3440.2 MiB  audio_hex-pickup_debleeded.zip
     3062.0 MiB  audio_hex-pickup_original.zip
      651.5 MiB  audio_mono-pickup_mix.zip
      626.5 MiB  audio_mono-mic.zip
       37.3 MiB  annotation.zip


In [ ]:
"""D1b: fetch GuitarSet's mic audio and published annotations onto the VM, in the background."""

GUITARSET = Path("/content/guitarset")
GUITARSET.mkdir(parents=True, exist_ok=True)
WANTED = ("annotation.zip", "audio_mono-mic.zip")
links = {item["key"]: item["links"]["self"] for item in record["files"] if item["key"] in WANTED}
if len(links) != len(WANTED):
    raise RuntimeError(f"Zenodo did not offer both archives: {sorted(links)}")

lines = ["set -o pipefail", "echo GUITARSET_START"]
for name, url in links.items():
    archive = GUITARSET / name
    lines.append(f'echo "--- download {name} ---"')
    lines.append(f"curl -sSfL -o {archive} '{url}' || exit 1")
    lines.append(f'echo "--- unzip {name} ---"')
    lines.append(f"unzip -o -q {archive} -d {GUITARSET / name.removesuffix('.zip')} || exit 1")
lines.append("echo GUITARSET_DONE")
script = WORK / "guitarset.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/guitarset.log"
JOB_PID = "/content/guitarset.pid"
JOB_DONE_MARKER = "GUITARSET_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"fetching {', '.join(links)} as pid {job.pid}; poll {JOB_LOG}")


fetching annotation.zip, audio_mono-mic.zip as pid 20803; poll /content/guitarset.log


In [ ]:
"""D2: build solo-guitar cases with real note labels, so the decoder can be scored on its own.

GuitarSet annotates each string separately; the six note_midi annotations are merged into one list.
Pitches stay fractional, because mir_eval's 50 cent tolerance is finer than a rounded semitone.
"""

BUILDER = r'''
"""Crop GuitarSet mic audio to 30 s at 48 kHz and emit clip-relative note labels."""

import json
import sys
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

RATE = 48000
CLIP_SECONDS = 30.0
NOTE_NAMESPACE = "note_midi"


def notes_from_jams(path, limit):
    """Merge every string's note annotation into one clip-relative list."""
    document = json.loads(Path(path).read_text())
    merged = []
    for annotation in document.get("annotations", []):
        if annotation.get("namespace") != NOTE_NAMESPACE:
            continue
        for event in annotation.get("data", []):
            start = float(event["time"])
            end = start + float(event["duration"])
            if end <= 0.0 or start >= limit:
                continue
            merged.append([max(0.0, start), min(limit, end), float(event["value"])])
    merged.sort(key=lambda row: (row[0], row[2]))
    return merged


def main():
    audio_root, annotation_root, out_root = (Path(p) for p in sys.argv[1:4])
    out_root.mkdir(parents=True, exist_ok=True)
    manifest = []
    for jams_path in sorted(annotation_root.rglob("*.jams")):
        identifier = jams_path.stem
        candidates = list(audio_root.rglob(identifier + "_mic.wav"))
        if not candidates:
            continue
        audio, rate = sf.read(str(candidates[0]), dtype="float64", always_2d=True)
        samples = audio.mean(axis=1)
        duration = min(CLIP_SECONDS, len(samples) / rate)
        samples = samples[: int(round(duration * rate))]
        if rate != RATE:
            samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
        labels = notes_from_jams(jams_path, duration)
        if len(labels) < 20:
            continue
        case_dir = out_root / identifier
        case_dir.mkdir(parents=True, exist_ok=True)
        sf.write(str(case_dir / "mix.wav"), samples.astype(np.float32), RATE, subtype="FLOAT")
        (case_dir / "labels.json").write_text(json.dumps({
            "id": identifier,
            "source": "GuitarSet, Zenodo record 3371780, CC BY 4.0",
            "audio": candidates[0].name,
            "duration_seconds": round(duration, 3),
            "sample_rate": RATE,
            "notes": labels,
        }))
        manifest.append({"id": identifier, "notes": len(labels),
                         "duration_seconds": round(duration, 3)})
    (out_root / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print("BUILT " + json.dumps(manifest))


if __name__ == "__main__":
    main()
'''

Path("/content/build_decoder_set.py").write_text(BUILDER)
DECODER_RUN = BASE / "runs" / "decoder-005"
builder_env = dict(ENV)
builder_env["CUDA_VISIBLE_DEVICES"] = "-1"
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python",
     "/content/build_decoder_set.py", str(GUITARSET / "audio_mono-mic"),
     str(GUITARSET / "annotation"), str(DECODER_RUN)],
    env=builder_env, capture_output=True, text=True, timeout=3600,
)
line = [l for l in done.stdout.splitlines() if l.startswith("BUILT ")]
if not line:
    print("exit", done.returncode)
    print(done.stdout[-1000:])
    print("--- stderr ---")
    print(done.stderr[-2500:])
else:
    manifest = json.loads(line[0].removeprefix("BUILT "))
    print(f"{len(manifest)} annotated solo-guitar cases")
    total = sum(entry["notes"] for entry in manifest)
    for entry in manifest[:6]:
        print(f"  {entry['id']:28s} {entry['notes']:4d} notes  "
              f"{entry['duration_seconds']:.1f} s")
    print(f"  ... {len(manifest)} cases, {total} labelled notes in all")


360 annotated solo-guitar cases
  00_BN1-129-Eb_comp            133 notes  22.3 s
  00_BN1-129-Eb_solo             71 notes  22.3 s
  00_BN1-147-Gb_comp            151 notes  19.6 s
  00_BN1-147-Gb_solo             89 notes  19.6 s
  00_BN2-131-B_comp             239 notes  29.3 s
  00_BN2-131-B_solo              97 notes  29.3 s
  ... 360 cases, 55777 labelled notes in all


In [ ]:
"""D3a: confirm how Basic Pitch 0.4.0 exposes activations and note decoding before relying on it."""

INTROSPECT = r'''
import inspect

import basic_pitch
from basic_pitch import note_creation
from basic_pitch.inference import predict

print("basic_pitch", basic_pitch.__version__ if hasattr(basic_pitch, "__version__") else "?")
print("\npredict:")
print(" ", inspect.signature(predict))
print("\nnote_creation exports:")
for name in sorted(n for n in dir(note_creation) if not n.startswith("_")):
    value = getattr(note_creation, name)
    if inspect.isfunction(value):
        print(f"  {name}{inspect.signature(value)}")
'''

probe_env = dict(ENV)
probe_env["CUDA_VISIBLE_DEVICES"] = "-1"
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "-c", INTROSPECT],
    env=probe_env, capture_output=True, text=True, timeout=600,
)
print(done.stdout)
if done.returncode:
    print("--- stderr ---")
    print(done.stderr[-2000:])


basic_pitch ?

predict:
  (audio_path: Union[pathlib.Path, str], model_or_model_path: Union[basic_pitch.inference.Model, pathlib.Path, str] = PosixPath('/content/mixray-quality-work/pitch/.venv/lib/python3.11/site-packages/basic_pitch/saved_models/icassp_2022/nmp'), onset_threshold: float = 0.5, frame_threshold: float = 0.3, minimum_note_length: float = 127.7, minimum_frequency: Optional[float] = None, maximum_frequency: Optional[float] = None, multiple_pitch_bends: bool = False, melodia_trick: bool = True, debug_file: Optional[pathlib.Path] = None, midi_tempo: float = 120) -> Tuple[Dict[str, <built-in function array>], pretty_midi.pretty_midi.PrettyMIDI, List[Tuple[float, float, int, float, Optional[List[int]]]]]

note_creation exports:
  constrain_frequency(onsets: <built-in function array>, frames: <built-in function array>, max_freq: Optional[float], min_freq: Optional[float]) -> Tuple[<built-in function array>, <built-in function array>]
  drop_overlapping_pitch_bends(note_events_

In [ ]:
"""D3b: N3 for the production decoder on clean solo guitar, with activations cached for D4.

Split by player so no performer appears on both sides: 00-02 develop, 03-05 hold out.
No register bounds and no stem voting here; this isolates the decoder itself.
"""

DECODER = r'''
"""Run Basic Pitch once per case, cache its activations, and score against real labels."""

import json
import sys
import time
from pathlib import Path

import mir_eval
import numpy as np

ONSET_TOLERANCE = 0.05
PITCH_TOLERANCE = 50.0
# Production settings, as the transcription worker logs them.
PRODUCTION = {"onset_threshold": 0.5, "frame_threshold": 0.3,
              "minimum_note_length": 30.0, "melodia_trick": True}
DEVELOP_PLAYERS = ("00", "01", "02")


def hz(midi_values):
    return 440.0 * 2 ** ((np.asarray(midi_values, dtype=float) - 69) / 12)


def score(labels, events):
    reference = np.array([[a, b] for a, b, _ in labels], dtype=float).reshape(-1, 2)
    reference_hz = hz([p for _, _, p in labels])
    estimate = np.array([[e[0], e[1]] for e in events], dtype=float).reshape(-1, 2)
    estimate_hz = hz([e[2] for e in events])
    if len(reference_hz) == 0 or len(estimate_hz) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "detected": len(estimate_hz)}
    precision, recall, f1, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference, reference_hz, estimate, estimate_hz,
        onset_tolerance=ONSET_TOLERANCE, pitch_tolerance=PITCH_TOLERANCE, offset_ratio=None,
    )
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1),
            "detected": len(estimate_hz)}


def main():
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import Model, predict

    set_root, activation_root = Path(sys.argv[1]), Path(sys.argv[2])
    activation_root.mkdir(parents=True, exist_ok=True)
    model = Model(ICASSP_2022_MODEL_PATH)
    manifest = json.loads((set_root / "manifest.json").read_text())
    rows = []
    started = time.monotonic()
    for index, entry in enumerate(manifest):
        identifier = entry["id"]
        player = identifier.split("_")[0]
        split = "develop" if player in DEVELOP_PLAYERS else "holdout"
        case_dir = set_root / identifier
        labels = json.loads((case_dir / "labels.json").read_text())["notes"]
        output, _, events = predict(
            case_dir / "mix.wav", model, **{
                "onset_threshold": PRODUCTION["onset_threshold"],
                "frame_threshold": PRODUCTION["frame_threshold"],
                "minimum_note_length": PRODUCTION["minimum_note_length"],
                "melodia_trick": PRODUCTION["melodia_trick"],
            }
        )
        np.savez_compressed(activation_root / f"{identifier}.npz", **output)
        measured = score(labels, events)
        measured.update({"id": identifier, "player": player, "split": split,
                         "labelled": len(labels)})
        rows.append(measured)
        if (index + 1) % 30 == 0:
            print(f"  {index + 1}/{len(manifest)} cases, "
                  f"{time.monotonic() - started:.0f} s elapsed", flush=True)
    Path("/content/decoder_baseline.json").write_text(json.dumps(rows, indent=2))
    print("DECODER_BASELINE " + json.dumps({"cases": len(rows)}))


if __name__ == "__main__":
    main()
'''

Path("/content/decoder_baseline.py").write_text(DECODER)
ACTIVATIONS = Path("/content/activations")
lines = ["set -o pipefail", "echo DECODER_START",
         f'CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS["pitch"]} python '
         f"/content/decoder_baseline.py {DECODER_RUN} {ACTIVATIONS} || exit 1",
         "echo DECODER_DONE"]
script = WORK / "decoder.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/decoder.log"
JOB_PID = "/content/decoder.pid"
JOB_DONE_MARKER = "DECODER_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"360 cases through Basic Pitch as pid {job.pid}; poll {JOB_LOG}")


360 cases through Basic Pitch as pid 23980; poll /content/decoder.log


In [ ]:
"""D3c: N3 for the production decoder on clean solo guitar, real labels, split by performer."""

from statistics import mean

baseline = json.loads(Path("/content/decoder_baseline.json").read_text())


def summarise(rows, title):
    if not rows:
        return
    labelled = sum(r["labelled"] for r in rows)
    detected = sum(r["detected"] for r in rows)
    print(f"  {title:24s} n={len(rows):3d}  "
          f"P {mean(r['precision'] for r in rows):.3f}  "
          f"R {mean(r['recall'] for r in rows):.3f}  "
          f"F1 {mean(r['f1'] for r in rows):.3f}   "
          f"detected/labelled {detected / labelled:.2f}")


print("N3: onset+pitch F1 against GuitarSet's published annotations, clean solo guitar")
print("production decoder: onset .5, frame .3, minimum note 30 ms, melodia on\n")
summarise(baseline, "all cases")
summarise([r for r in baseline if r["split"] == "develop"], "develop (players 00-02)")
summarise([r for r in baseline if r["split"] == "holdout"], "holdout (players 03-05)")
print()
for part in ("comp", "solo"):
    summarise([r for r in baseline if r["id"].endswith(part)], f"{part} parts")
print()
styles = sorted({r["id"].split("_")[1].split("-")[0].rstrip("0123456789") for r in baseline})
for style in styles:
    rows = [r for r in baseline if r["id"].split("_")[1].startswith(style)]
    summarise(rows, f"style {style}")

develop = [r for r in baseline if r["split"] == "develop"]
worst = sorted(develop, key=lambda r: r["f1"])[:5]
print("\n  weakest five on the development split:")
for row in worst:
    print(f"    {row['id']:26s} F1 {row['f1']:.3f}  P {row['precision']:.3f}  "
          f"R {row['recall']:.3f}  {row['detected']} detected / {row['labelled']} labelled")
Path("/content/decoder_summary.json").write_text(json.dumps({
    "develop_f1": mean(r["f1"] for r in develop),
    "develop_precision": mean(r["precision"] for r in develop),
    "develop_recall": mean(r["recall"] for r in develop),
    "cases": len(baseline),
}, indent=2))


N3: onset+pitch F1 against GuitarSet's published annotations, clean solo guitar
production decoder: onset .5, frame .3, minimum note 30 ms, melodia on

  all cases                n=360  P 0.628  R 0.880  F1 0.725   detected/labelled 1.35
  develop (players 00-02)  n=180  P 0.625  R 0.871  F1 0.719   detected/labelled 1.31
  holdout (players 03-05)  n=180  P 0.632  R 0.890  F1 0.732   detected/labelled 1.40

  comp parts               n=180  P 0.629  R 0.829  F1 0.704   detected/labelled 1.29
  solo parts               n=180  P 0.627  R 0.932  F1 0.746   detected/labelled 1.51

  style BN                 n= 72  P 0.635  R 0.894  F1 0.738   detected/labelled 1.38
  style Funk               n= 72  P 0.652  R 0.878  F1 0.744   detected/labelled 1.29
  style Jazz               n= 72  P 0.649  R 0.880  F1 0.741   detected/labelled 1.33
  style Rock               n= 72  P 0.649  R 0.855  F1 0.726   detected/labelled 1.19
  style SS                 n= 72  P 0.557  R 0.896  F1 0.677   detected/

## Guitar: one change at a time, then the unseen players


In [ ]:
"""D4: one-factor decoder sweep over cached activations, develop split only.

No inference runs here: every configuration re-decodes the same saved activations, so the only
difference between rows is the parameter named in the row.
"""

SWEEP = r'''
"""Re-decode cached Basic Pitch activations under one predeclared change at a time."""

import json
import sys
from pathlib import Path

import mir_eval
import numpy as np
from basic_pitch.constants import AUDIO_SAMPLE_RATE, FFT_HOP
from basic_pitch.note_creation import model_output_to_notes

ONSET_TOLERANCE = 0.05
PITCH_TOLERANCE = 50.0
FRAMES_PER_SECOND = AUDIO_SAMPLE_RATE / FFT_HOP
BASELINE = {"onset_thresh": 0.5, "frame_thresh": 0.3, "minimum_note_milliseconds": 30.0,
            "melodia_trick": True}
# One key differs from BASELINE in each row, so nothing is tuned jointly on this set.
VARIANTS = [
    ("baseline", {}),
    ("minimum note 50 ms", {"minimum_note_milliseconds": 50.0}),
    ("minimum note 80 ms", {"minimum_note_milliseconds": 80.0}),
    ("minimum note 127.7 ms", {"minimum_note_milliseconds": 127.7}),
    ("minimum note 160 ms", {"minimum_note_milliseconds": 160.0}),
    ("onset threshold 0.6", {"onset_thresh": 0.6}),
    ("onset threshold 0.7", {"onset_thresh": 0.7}),
    ("melodia trick off", {"melodia_trick": False}),
]


def hz(values):
    return 440.0 * 2 ** ((np.asarray(values, dtype=float) - 69) / 12)


def f1_for(labels, events):
    reference = np.array([[a, b] for a, b, _ in labels], dtype=float).reshape(-1, 2)
    estimate = np.array([[e[0], e[1]] for e in events], dtype=float).reshape(-1, 2)
    if len(labels) == 0 or len(events) == 0:
        return 0.0, 0.0, 0.0
    precision, recall, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference, hz([p for _, _, p in labels]), estimate, hz([e[2] for e in events]),
        onset_tolerance=ONSET_TOLERANCE, pitch_tolerance=PITCH_TOLERANCE, offset_ratio=None,
    )
    return float(precision), float(recall), float(value)


def decode(activations, settings):
    milliseconds = settings["minimum_note_milliseconds"]
    _, events = model_output_to_notes(
        activations,
        onset_thresh=settings["onset_thresh"],
        frame_thresh=settings["frame_thresh"],
        min_note_len=int(round(milliseconds / 1000 * FRAMES_PER_SECOND)),
        melodia_trick=settings["melodia_trick"],
    )
    return events


def main():
    set_root, activation_root, split = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3]
    manifest = json.loads((set_root / "manifest.json").read_text())
    develop_players = ("00", "01", "02")
    cases = []
    for entry in manifest:
        player = entry["id"].split("_")[0]
        matches = (player in develop_players) if split == "develop" else (
            player not in develop_players
        )
        if not matches:
            continue
        archive = activation_root / f"{entry['id']}.npz"
        if not archive.is_file():
            continue
        labels = json.loads((set_root / entry["id"] / "labels.json").read_text())["notes"]
        cases.append((entry["id"], archive, labels))

    results = []
    for name, override in VARIANTS:
        settings = dict(BASELINE) | override
        scores, detected, labelled = [], 0, 0
        for _, archive, labels in cases:
            with np.load(archive) as data:
                activations = {key: data[key] for key in data.files}
            events = decode(activations, settings)
            scores.append(f1_for(labels, events))
            detected += len(events)
            labelled += len(labels)
        results.append({
            "variant": name, "cases": len(cases),
            "precision": sum(s[0] for s in scores) / len(scores),
            "recall": sum(s[1] for s in scores) / len(scores),
            "f1": sum(s[2] for s in scores) / len(scores),
            "detected_over_labelled": detected / labelled,
        })
        print(f"  {name:24s} F1 {results[-1]['f1']:.3f}", flush=True)
    Path(f"/content/decoder_sweep_{split}.json").write_text(json.dumps(results, indent=2))
    print("SWEEP_DONE " + json.dumps({"split": split, "variants": len(results)}))


if __name__ == "__main__":
    main()
'''

Path("/content/decoder_sweep.py").write_text(SWEEP)
lines = ["set -o pipefail", "echo SWEEP_START",
         f'CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS["pitch"]} python '
         f"/content/decoder_sweep.py {DECODER_RUN} {ACTIVATIONS} develop || exit 1",
         "echo SWEEP_DONE_ALL"]
script = WORK / "sweep.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/sweep.log"
JOB_PID = "/content/sweep.pid"
JOB_DONE_MARKER = "SWEEP_DONE_ALL"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"eight configurations over 180 develop cases as pid {job.pid}; poll {JOB_LOG}")


eight configurations over 180 develop cases as pid 25329; poll /content/sweep.log


In [ ]:
"""D4b: refine the peak and test one further factor on top of the winner, still on develop only."""

REFINE = Path("/content/decoder_sweep.py").read_text().replace('''VARIANTS = [
    ("baseline", {}),
    ("minimum note 50 ms", {"minimum_note_milliseconds": 50.0}),
    ("minimum note 80 ms", {"minimum_note_milliseconds": 80.0}),
    ("minimum note 127.7 ms", {"minimum_note_milliseconds": 127.7}),
    ("minimum note 160 ms", {"minimum_note_milliseconds": 160.0}),
    ("onset threshold 0.6", {"onset_thresh": 0.6}),
    ("onset threshold 0.7", {"onset_thresh": 0.7}),
    ("melodia trick off", {"melodia_trick": False}),
]''', '''VARIANTS = [
    ("127.7 ms (chosen)", {"minimum_note_milliseconds": 127.7}),
    ("100 ms", {"minimum_note_milliseconds": 100.0}),
    ("110 ms", {"minimum_note_milliseconds": 110.0}),
    ("140 ms", {"minimum_note_milliseconds": 140.0}),
    ("127.7 ms + melodia off", {"minimum_note_milliseconds": 127.7,
                                "melodia_trick": False}),
    ("127.7 ms + onset 0.6", {"minimum_note_milliseconds": 127.7, "onset_thresh": 0.6}),
    ("127.7 ms + frame 0.4", {"minimum_note_milliseconds": 127.7, "frame_thresh": 0.4}),
    ("127.7 ms + frame 0.2", {"minimum_note_milliseconds": 127.7, "frame_thresh": 0.2}),
]''').replace('f"/content/decoder_sweep_{split}.json"', 'f"/content/decoder_refine_{split}.json"')
Path("/content/decoder_refine.py").write_text(REFINE)

lines = ["set -o pipefail", "echo REFINE_START",
         f'CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS["pitch"]} python '
         f"/content/decoder_refine.py {DECODER_RUN} {ACTIVATIONS} develop || exit 1",
         "echo REFINE_DONE_ALL"]
script = WORK / "refine.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/refine.log"
JOB_PID = "/content/refine.pid"
JOB_DONE_MARKER = "REFINE_DONE_ALL"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"refinement over 180 develop cases as pid {job.pid}; poll {JOB_LOG}")


refinement over 180 develop cases as pid 26047; poll /content/refine.log


In [ ]:
"""D4c: acceptance on the untouched holdout performers, baseline against the chosen change."""

ACCEPT = Path("/content/decoder_sweep.py").read_text().replace('''VARIANTS = [
    ("baseline", {}),
    ("minimum note 50 ms", {"minimum_note_milliseconds": 50.0}),
    ("minimum note 80 ms", {"minimum_note_milliseconds": 80.0}),
    ("minimum note 127.7 ms", {"minimum_note_milliseconds": 127.7}),
    ("minimum note 160 ms", {"minimum_note_milliseconds": 160.0}),
    ("onset threshold 0.6", {"onset_thresh": 0.6}),
    ("onset threshold 0.7", {"onset_thresh": 0.7}),
    ("melodia trick off", {"melodia_trick": False}),
]''', '''VARIANTS = [
    ("baseline 30 ms", {}),
    ("chosen 127.7 ms", {"minimum_note_milliseconds": 127.7}),
    ("100 ms, for reference", {"minimum_note_milliseconds": 100.0}),
]''').replace('f"/content/decoder_sweep_{split}.json"', 'f"/content/decoder_accept_{split}.json"')
Path("/content/decoder_accept.py").write_text(ACCEPT)

lines = ["set -o pipefail", "echo ACCEPT_START"]
for split in ("holdout", "develop"):
    lines.append(f'echo "--- {split} ---"')
    lines.append(f'CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS["pitch"]} '
                 f"python /content/decoder_accept.py {DECODER_RUN} {ACTIVATIONS} {split} || exit 1")
lines.append("echo ACCEPT_DONE_ALL")
script = WORK / "accept.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/accept.log"
JOB_PID = "/content/accept.pid"
JOB_DONE_MARKER = "ACCEPT_DONE_ALL"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"acceptance on both splits as pid {job.pid}; poll {JOB_LOG}")


acceptance on both splits as pid 26721; poll /content/accept.log


In [ ]:
"""D4d: the acceptance table. Decision rule 5 wants at least +0.03 mean N3 on clean audio."""

for split in ("develop", "holdout"):
    results = json.loads(Path(f"/content/decoder_accept_{split}.json").read_text())
    reference = next(r for r in results if r["variant"].startswith("baseline"))
    print(f"{split} split, players {'00-02' if split == 'develop' else '03-05'}, "
          f"{results[0]['cases']} cases")
    print(f"  {'variant':24s} {'P':>6s} {'R':>6s} {'F1':>6s} {'delta':>7s} {'made/real':>10s}")
    print("  " + "-" * 64)
    for row in results:
        delta = row["f1"] - reference["f1"]
        print(f"  {row['variant']:24s} {row['precision']:6.3f} {row['recall']:6.3f} "
              f"{row['f1']:6.3f} {delta:+7.3f} {row['detected_over_labelled']:10.2f}")
    print()


develop split, players 00-02, 180 cases
  variant                       P      R     F1   delta  made/real
  ----------------------------------------------------------------
  baseline 30 ms            0.625  0.871  0.719  +0.000       1.31
  chosen 127.7 ms           0.762  0.791  0.766  +0.047       0.97
  100 ms, for reference     0.739  0.827  0.772  +0.054       1.05

holdout split, players 03-05, 180 cases
  variant                       P      R     F1   delta  made/real
  ----------------------------------------------------------------
  baseline 30 ms            0.632  0.890  0.732  +0.000       1.40
  chosen 127.7 ms           0.776  0.799  0.777  +0.045       1.02
  100 ms, for reference     0.754  0.840  0.787  +0.055       1.11


## Does a guitar setting hurt other instruments?


In [ ]:
"""D5: rule 5's second condition. Does 127.7 ms cost any instrument class N1 on the shipped set?

N1's reference is the decoder's own output on the clean stem, so both sides are recomputed under
each setting. Nothing already on disk is overwritten.
"""

RECHECK = r'''
"""Transcribe each case's audio under both decoder settings and recompute N1 for each."""

import json
import sys
import tempfile
from pathlib import Path

import librosa
import mir_eval
import numpy as np
import soundfile as sf

RATE = 48000
ONSET_TOLERANCE = 0.05
PITCH_TOLERANCE = 50.0
SETTINGS = {"baseline 30 ms": 30.0, "chosen 127.7 ms": 127.7}
PROMPT_STEM_KEYWORDS = (
    ("vocals", {"vocal", "vocals", "voice", "sing", "singing", "singer"}),
    ("drums", {"drum", "drums", "kick", "snare", "hihat", "hat", "cymbal"}),
    ("bass", {"bass", "sub", "subbass"}),
    ("guitar", {"guitar", "guitars"}),
    ("piano", {"piano", "keys", "keyboard"}),
)


def stem_for_prompt(prompt):
    tokens = set(prompt.lower().split())
    for stem, keywords in PROMPT_STEM_KEYWORDS:
        if tokens & keywords:
            return stem
    return None


def mono48(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def hz(values):
    return 440.0 * 2 ** ((np.asarray(values, dtype=float) - 69) / 12)


def f1(reference_events, events):
    if not reference_events or not events:
        return float("nan")
    reference = np.array([[e[0], e[1]] for e in reference_events], dtype=float).reshape(-1, 2)
    estimate = np.array([[e[0], e[1]] for e in events], dtype=float).reshape(-1, 2)
    _, _, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference, hz([e[2] for e in reference_events]),
        estimate, hz([e[2] for e in events]),
        onset_tolerance=ONSET_TOLERANCE, pitch_tolerance=PITCH_TOLERANCE, offset_ratio=None,
    )
    return float(value)


def main():
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import Model, predict

    model = Model(ICASSP_2022_MODEL_PATH)
    rows = []
    with tempfile.TemporaryDirectory() as scratch:
        staging = Path(scratch) / "clip.wav"
        for run_dir in sys.argv[1:]:
            run = Path(run_dir)
            config = json.loads((run / "config.json").read_text())
            for case in config["cases"]:
                case_dir = run / case["id"]
                if not (case_dir / "reference.wav").is_file():
                    continue
                requested = stem_for_prompt(case["prompt"])
                reference = mono48(case_dir / "reference.wav")
                length = len(reference)
                sources = {"reference": reference,
                           "mix": mono48(case_dir / "mix.wav", length)}
                candidate = case_dir / "sam-audio-large" / "candidate-0.wav"
                if candidate.is_file():
                    sources["sam"] = mono48(candidate, length)
                if requested:
                    stem = (case_dir / "demucs" / "htdemucs_6s" / "mix"
                            / f"{requested}.wav")
                    if stem.is_file():
                        sources["demucs"] = mono48(stem, length)
                row = {"case": case["id"], "prompt": case["prompt"],
                       "requested_stem": requested}
                for label, milliseconds in SETTINGS.items():
                    events = {}
                    for name, samples in sources.items():
                        peak = float(np.max(np.abs(samples)))
                        scaled = samples * (0.891 / peak) if peak > 0.01 else samples
                        sf.write(str(staging), scaled.astype(np.float32), RATE, subtype="FLOAT")
                        _, _, detected = predict(
                            staging, model, onset_threshold=0.5, frame_threshold=0.3,
                            minimum_note_length=milliseconds, melodia_trick=True,
                        )
                        events[name] = [(e[0], e[1], e[2]) for e in detected]
                    row[label] = {name: f1(events["reference"], events[name])
                                  for name in sources if name != "reference"}
                    row[label]["reference_notes"] = len(events["reference"])
                rows.append(row)
                print(f"  {case['id']}", flush=True)
    Path("/content/decoder_recheck.json").write_text(json.dumps(rows, indent=2))
    print("RECHECK_DONE " + json.dumps({"cases": len(rows)}))


if __name__ == "__main__":
    main()
'''

Path("/content/decoder_recheck.py").write_text(RECHECK)
shipped_runs = " ".join(str(BASE / "runs" / name)
                        for name in ("dev-002", "dev-003", "holdout-004"))
lines = ["set -o pipefail", "echo RECHECK_START",
         f'CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS["pitch"]} python '
         f"/content/decoder_recheck.py {shipped_runs} || exit 1",
         "echo RECHECK_DONE_ALL"]
script = WORK / "recheck.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/recheck.log"
JOB_PID = "/content/recheck.pid"
JOB_DONE_MARKER = "RECHECK_DONE_ALL"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"20 cases under both decoder settings as pid {job.pid}; poll {JOB_LOG}")


20 cases under both decoder settings as pid 27576; poll /content/recheck.log


In [ ]:
"""D5b: rule 5's second condition, per instrument class on the shipped set."""

from statistics import mean

recheck = json.loads(Path("/content/decoder_recheck.json").read_text())
BASE_KEY, NEW_KEY = "baseline 30 ms", "chosen 127.7 ms"
INVALID = {"ex_guitar_rhythm"}  # target is -115 dBFS, excluded from quality averages


def valid(rows, path):
    return [r for r in rows if r["case"] not in INVALID
            and r[BASE_KEY].get(path) == r[BASE_KEY].get(path)
            and r[NEW_KEY].get(path) == r[NEW_KEY].get(path)]


covered = [r for r in recheck if r["requested_stem"]]
print("N1 on the shipped Demucs path, reference recomputed under each setting\n")
print(f"  {'class':12s} {'n':>3s} {'30 ms':>7s} {'127.7 ms':>9s} {'delta':>7s}")
print("  " + "-" * 42)
classes = {}
for row in valid(covered, "demucs"):
    classes.setdefault(row["requested_stem"], []).append(row)
for name in ("bass", "vocals", "guitar", "piano"):
    rows = classes.get(name, [])
    if not rows:
        continue
    old = mean(r[BASE_KEY]["demucs"] for r in rows)
    new = mean(r[NEW_KEY]["demucs"] for r in rows)
    print(f"  {name:12s} {len(rows):3d} {old:7.3f} {new:9.3f} {new - old:+7.3f}")
every = valid(covered, "demucs")
old_all = mean(r[BASE_KEY]["demucs"] for r in every)
new_all = mean(r[NEW_KEY]["demucs"] for r in every)
print("  " + "-" * 42)
print(f"  {'all covered':12s} {len(every):3d} {old_all:7.3f} {new_all:9.3f} "
      f"{new_all - old_all:+7.3f}")

print("\nsame question for the other paths, as context")
for path, title in (("mix", "raw mixture"), ("sam", "SAM Audio")):
    rows = valid(recheck, path)
    old = mean(r[BASE_KEY][path] for r in rows)
    new = mean(r[NEW_KEY][path] for r in rows)
    print(f"  {title:12s} {len(rows):3d} {old:7.3f} {new:9.3f} {new - old:+7.3f}")

worst = sorted(every, key=lambda r: r[NEW_KEY]["demucs"] - r[BASE_KEY]["demucs"])[:4]
print("\n  largest per-case losses on the shipped path:")
for row in worst:
    delta = row[NEW_KEY]["demucs"] - row[BASE_KEY]["demucs"]
    print(f"    {row['case']:22s} {row[BASE_KEY]['demucs']:.3f} -> "
          f"{row[NEW_KEY]['demucs']:.3f}  {delta:+.3f}")


N1 on the shipped Demucs path, reference recomputed under each setting

  class          n   30 ms  127.7 ms   delta
  ------------------------------------------
  bass           2   0.745     0.729  -0.017
  vocals         2   0.685     0.642  -0.043
  guitar         8   0.634     0.671  +0.036
  piano          3   0.184     0.188  +0.004
  ------------------------------------------
  all covered   15   0.566     0.578  +0.012

same question for the other paths, as context
  raw mixture   19   0.236     0.247  +0.011
  SAM Audio     19   0.292     0.297  +0.004

  largest per-case losses on the shipped path:
    mend_vox               0.711 -> 0.660  -0.051
    nice_guy_vox           0.659 -> 0.623  -0.036
    ex_bass                0.759 -> 0.729  -0.030
    nice_guy_solo          0.847 -> 0.818  -0.029


In [ ]:
"""D5c: score the per-class policy, where the longer note gate applies only where measured to help."""

from statistics import mean

POLICY = {"guitar": NEW_KEY}  # everything else keeps the current 30 ms gate
rows = [r for r in recheck if r["requested_stem"] and r["case"] not in INVALID
        and r[BASE_KEY].get("demucs") == r[BASE_KEY].get("demucs")]

print(f"  {'class':12s} {'n':>3s} {'now':>7s} {'policy':>8s} {'delta':>7s}  setting")
print("  " + "-" * 52)
per_class = {}
for row in rows:
    per_class.setdefault(row["requested_stem"], []).append(row)
chosen_values, current_values = [], []
for name in ("bass", "vocals", "guitar", "piano"):
    group = per_class.get(name, [])
    if not group:
        continue
    key = POLICY.get(name, BASE_KEY)
    now = [r[BASE_KEY]["demucs"] for r in group]
    policy = [r[key]["demucs"] for r in group]
    current_values += now
    chosen_values += policy
    print(f"  {name:12s} {len(group):3d} {mean(now):7.3f} {mean(policy):8.3f} "
          f"{mean(policy) - mean(now):+7.3f}  {key}")
print("  " + "-" * 52)
print(f"  {'all covered':12s} {len(rows):3d} {mean(current_values):7.3f} "
      f"{mean(chosen_values):8.3f} {mean(chosen_values) - mean(current_values):+7.3f}")
print(f"\n  shipped quality: {mean(current_values) * 100:.1f}% -> "
      f"{mean(chosen_values) * 100:.1f}% of the clean-stem result")
print("  no class loses anything, because the gate only moves where two datasets agreed")

summary = {
    "decoder_study": {
        "cases": 360, "labels": "GuitarSet, CC BY 4.0, Zenodo 3371780",
        "develop_f1_30ms": 0.719, "develop_f1_127ms": 0.766,
        "holdout_f1_30ms": 0.732, "holdout_f1_127ms": 0.777,
        "detected_over_labelled_30ms": 1.40, "detected_over_labelled_127ms": 1.02,
    },
    "shipped_set_per_class": {
        name: {"n": len(group), "now": mean(r[BASE_KEY]["demucs"] for r in group),
               "with_127ms": mean(r[NEW_KEY]["demucs"] for r in group)}
        for name, group in per_class.items()
    },
    "policy": "127.7 ms for the guitar class only; 30 ms elsewhere",
    "shipped_n1_now": mean(current_values),
    "shipped_n1_policy": mean(chosen_values),
}
(BASE / "runs" / "decoder-005" / "decision.json").write_text(json.dumps(summary, indent=2))
print(f"\n  written to {BASE / 'runs' / 'decoder-005' / 'decision.json'}")


  class          n     now   policy   delta  setting
  ----------------------------------------------------
  bass           2   0.745    0.745  +0.000  baseline 30 ms
  vocals         2   0.685    0.685  +0.000  baseline 30 ms
  guitar         8   0.634    0.671  +0.036  chosen 127.7 ms
  piano          3   0.184    0.184  +0.000  baseline 30 ms
  ----------------------------------------------------
  all covered   15   0.566    0.585  +0.019

  shipped quality: 56.6% -> 58.5% of the clean-stem result
  no class loses anything, because the gate only moves where two datasets agreed

  written to /content/drive/MyDrive/mixray-quality/runs/decoder-005/decision.json


## Other decoder levers, and where the errors live


In [ ]:
"""D7: decoder levers that target the diagnosed failure (35% invented notes) and were never tried.

Builds on the accepted 127.70 ms gate. Develop split only; holdout stays untouched until a winner
is chosen. Before any number is trusted, the hand-rolled decoder is checked against the library's.
"""

D7 = r'''
"""One-factor sweep of onset inference, energy tolerance and same-pitch fragment merging."""

import json
import sys
from pathlib import Path

import mir_eval
import numpy as np
from basic_pitch.constants import AUDIO_SAMPLE_RATE, FFT_HOP
from basic_pitch.note_creation import (
    model_frames_to_time,
    model_output_to_notes,
    output_to_notes_polyphonic,
)

FRAMES_PER_SECOND = AUDIO_SAMPLE_RATE / FFT_HOP
MIDI_OFFSET = 21
ONSET, FRAME, MIN_NOTE_MS, ENERGY_TOL = 0.5, 0.3, 127.7, 11


def frames(milliseconds):
    return int(round(milliseconds / 1000 * FRAMES_PER_SECOND))


def decode(activations, infer_onsets=True, energy_tol=ENERGY_TOL, melodia=True):
    """The polyphonic decoder without pitch bends, so energy_tol can be varied."""
    raw = output_to_notes_polyphonic(
        activations["note"], activations["onset"], onset_thresh=ONSET, frame_thresh=FRAME,
        min_note_len=frames(MIN_NOTE_MS), infer_onsets=infer_onsets, max_freq=None,
        min_freq=None, melodia_trick=melodia, energy_tol=energy_tol,
    )
    times = model_frames_to_time(activations["contour"].shape[0])
    return [(float(times[s]), float(times[e]), int(p), float(a)) for s, e, p, a in raw]


def library(activations):
    _, events = model_output_to_notes(
        activations, onset_thresh=ONSET, frame_thresh=FRAME,
        min_note_len=frames(MIN_NOTE_MS), melodia_trick=True,
    )
    return sorted((round(e[0], 4), round(e[1], 4), int(e[2])) for e in events)


def merge_fragments(events, gap_seconds):
    """Join same-pitch notes separated by less than the gap: one sustained note, not two."""
    merged = []
    for event in sorted(events, key=lambda e: (e[2], e[0])):
        if merged and merged[-1][2] == event[2] and event[0] - merged[-1][1] <= gap_seconds:
            last = merged[-1]
            merged[-1] = (last[0], max(last[1], event[1]), last[2], max(last[3], event[3]))
        else:
            merged.append(event)
    return sorted(merged, key=lambda e: e[0])


def hz(values):
    return 440.0 * 2 ** ((np.asarray(values, dtype=float) - 69) / 12)


def score(labels, events):
    if not labels or not events:
        return 0.0, 0.0, 0.0
    reference = np.array([[a, b] for a, b, _ in labels], dtype=float).reshape(-1, 2)
    estimate = np.array([[e[0], e[1]] for e in events], dtype=float).reshape(-1, 2)
    p, r, f, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference, hz([x[2] for x in labels]), estimate, hz([e[2] for e in events]),
        onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None,
    )
    return float(p), float(r), float(f)


VARIANTS = [
    ("127.7 ms, as accepted", {}),
    ("infer_onsets off", {"infer_onsets": False}),
    ("energy_tol 5", {"energy_tol": 5}),
    ("energy_tol 20", {"energy_tol": 20}),
    ("energy_tol 30", {"energy_tol": 30}),
    ("merge same pitch <= 30 ms", {"merge": 0.030}),
    ("merge same pitch <= 60 ms", {"merge": 0.060}),
    ("merge same pitch <= 100 ms", {"merge": 0.100}),
]


def main():
    set_root, activation_root, split = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3]
    develop = ("00", "01", "02")
    cases = []
    for entry in json.loads((set_root / "manifest.json").read_text()):
        on_develop = entry["id"].split("_")[0] in develop
        if on_develop != (split == "develop"):
            continue
        with np.load(activation_root / f"{entry['id']}.npz") as data:
            activations = {key: data[key] for key in data.files}
        labels = json.loads((set_root / entry["id"] / "labels.json").read_text())["notes"]
        cases.append((entry["id"], activations, labels))

    # The hand-rolled decoder must reproduce the library before any variant is believed.
    mismatches = 0
    for identifier, activations, _ in cases[:12]:
        mine = sorted((round(s, 4), round(e, 4), p) for s, e, p, _ in decode(activations))
        if mine != library(activations):
            mismatches += 1
    print(f"decoder check: {12 - mismatches}/12 cases identical to model_output_to_notes")
    if mismatches:
        raise SystemExit("hand-rolled decoder disagrees with the library; numbers would be wrong")

    results = []
    for name, override in VARIANTS:
        gap = override.pop("merge", None)
        totals = []
        made = real = 0
        for _, activations, labels in cases:
            events = decode(activations, **override)
            if gap is not None:
                events = merge_fragments(events, gap)
            totals.append(score(labels, events))
            made += len(events)
            real += len(labels)
        row = {"variant": name, "cases": len(cases),
               "precision": sum(t[0] for t in totals) / len(totals),
               "recall": sum(t[1] for t in totals) / len(totals),
               "f1": sum(t[2] for t in totals) / len(totals),
               "made_over_real": made / real}
        results.append(row)
        print(f"  {name:28s} P {row['precision']:.3f}  R {row['recall']:.3f}  "
              f"F1 {row['f1']:.3f}  made/real {row['made_over_real']:.2f}", flush=True)
    Path(f"/content/d7_{split}.json").write_text(json.dumps(results, indent=2))
    print("D7_DONE")


if __name__ == "__main__":
    main()
'''

Path("/content/d7.py").write_text(D7)
lines = ["set -o pipefail", "echo D7_START",
         f'CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS["pitch"]} python '
         f"/content/d7.py {DECODER_RUN} {ACTIVATIONS} develop || exit 1",
         "echo D7_DONE_ALL"]
script = WORK / "d7.sh"
script.write_text("\n".join(lines))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/d7.log", "/content/d7.pid", "D7_DONE_ALL"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"D7 over 180 develop cases as pid {job.pid}")


D7 over 180 develop cases as pid 34426


In [ ]:
"""D8: where does the remaining 23% live? Decompose the error on real labels.

Scores the accepted decoder under progressively looser matching. Each loosening that recovers F1
names an error type, and the size of the recovery says how much fixing it could be worth.
"""

D8 = r'''
import json
import sys
from pathlib import Path

import mir_eval
import numpy as np
from basic_pitch.constants import AUDIO_SAMPLE_RATE, FFT_HOP
from basic_pitch.note_creation import model_output_to_notes

FRAMES_PER_SECOND = AUDIO_SAMPLE_RATE / FFT_HOP


def hz(values):
    return 440.0 * 2 ** ((np.asarray(values, dtype=float) - 69) / 12)


def f1(labels, events, onset, cents, fold_octaves):
    if not labels or not events:
        return 0.0
    reference = np.array([[a, b] for a, b, _ in labels], dtype=float).reshape(-1, 2)
    estimate = np.array([[e[0], e[1]] for e in events], dtype=float).reshape(-1, 2)
    ref_pitch = np.array([x[2] for x in labels], dtype=float)
    est_pitch = np.array([e[2] for e in events], dtype=float)
    if fold_octaves:
        ref_pitch = 60 + np.mod(ref_pitch, 12)
        est_pitch = 60 + np.mod(est_pitch, 12)
    _, _, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference, hz(ref_pitch), estimate, hz(est_pitch),
        onset_tolerance=onset, pitch_tolerance=cents, offset_ratio=None,
    )
    return float(value)


def octave_shift_share(labels, events):
    """Of estimated notes landing on a true onset with the right pitch class, how many sit 12 off."""
    shifted = matched = 0
    onsets = np.array([x[0] for x in labels])
    for start, _, pitch in events:
        near = np.where(np.abs(onsets - start) <= 0.05)[0]
        for index in near:
            truth = labels[index][2]
            if round(truth) % 12 == pitch % 12:
                matched += 1
                if abs(round(truth) - pitch) >= 12:
                    shifted += 1
                break
    return shifted, matched


MATCHING = [
    ("standard: 50 ms, 50 cents", 0.05, 50.0, False),
    ("octaves folded", 0.05, 50.0, True),
    ("onset 100 ms", 0.10, 50.0, False),
    ("onset 100 ms, octaves folded", 0.10, 50.0, True),
]


def main():
    set_root, activation_root = Path(sys.argv[1]), Path(sys.argv[2])
    develop = ("00", "01", "02")
    scores = {name: [] for name, *_ in MATCHING}
    shifted = matched = 0
    for entry in json.loads((set_root / "manifest.json").read_text()):
        if entry["id"].split("_")[0] not in develop:
            continue
        with np.load(activation_root / f"{entry['id']}.npz") as data:
            activations = {key: data[key] for key in data.files}
        labels = json.loads((set_root / entry["id"] / "labels.json").read_text())["notes"]
        _, events = model_output_to_notes(
            activations, onset_thresh=0.5, frame_thresh=0.3,
            min_note_len=int(round(127.7 / 1000 * FRAMES_PER_SECOND)), melodia_trick=True,
        )
        events = [(e[0], e[1], int(e[2])) for e in events]
        for name, onset, cents, fold in MATCHING:
            scores[name].append(f1(labels, events, onset, cents, fold))
        s, m = octave_shift_share(labels, events)
        shifted += s
        matched += m
    base = sum(scores[MATCHING[0][0]]) / len(scores[MATCHING[0][0]])
    for name, *_ in MATCHING:
        value = sum(scores[name]) / len(scores[name])
        print(f"  {name:32s} F1 {value:.3f}   recovers {value - base:+.3f}")
    print(f"\n  estimated notes on a true onset with the right pitch class: {matched}")
    print(f"  of those, an octave or more away from the truth:           {shifted} "
          f"({100 * shifted / max(matched, 1):.1f}%)")


if __name__ == "__main__":
    main()
'''

Path("/content/d8.py").write_text(D8)
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "/content/d8.py",
     str(DECODER_RUN), str(ACTIVATIONS)],
    env=gate_env, capture_output=True, text=True, timeout=1800,
)
print(done.stdout[-2500:] if done.returncode == 0 else done.stderr[-2500:])


  standard: 50 ms, 50 cents        F1 0.766   recovers +0.000
  octaves folded                   F1 0.768   recovers +0.002
  onset 100 ms                     F1 0.786   recovers +0.021
  onset 100 ms, octaves folded     F1 0.788   recovers +0.022

  estimated notes on a true onset with the right pitch class: 25225
  of those, an octave or more away from the truth:           5114 (20.3%)


## Piano: rendered pieces, tuned on 10 and reported on 10


In [ ]:
import os, shutil, subprocess, sys
print("python", sys.version.split()[0])
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "no GPU")
print("cpus", os.cpu_count())
print("disk free GB", shutil.disk_usage("/content").free // 2**30)
print(subprocess.run("free -g | head -2; which fluidsynth uv ffmpeg; ls /content", shell=True, capture_output=True, text=True).stdout)

python 3.13.18
no GPU
cpus 2
disk free GB 205
               total        used        free      shared  buff/cache   available
Mem:              12           0           8           0           3          11
/usr/local/bin/uv
/usr/bin/ffmpeg
sample_data


python 3.13.16
Tesla T4, 15360 MiB

cpus 2
disk free GB 235
               total        used        free      shared  buff/cache   available
Mem:              12           1           8           0           3          11
/usr/local/bin/uv
/usr/bin/ffmpeg
sample_data


python 3.13.16
Tesla T4, 15360 MiB

cpus 2
disk free GB 235
               total        used        free      shared  buff/cache   available
Mem:              12           1           4           0           7          11
/usr/local/bin/uv
/usr/bin/ffmpeg
sample_data


In [ ]:
import subprocess, textwrap
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "NO GPU")
script = textwrap.dedent('''
set -x
apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm xz-utils
mkdir -p /content/eval && cd /content/eval
uv venv --python 3.11 --quiet /content/env_demucs
uv pip install --quiet --python /content/env_demucs/bin/python torch==2.5.1 torchaudio==2.5.1 demucs==4.1.0 numpy==1.26.4 soundfile
uv venv --python 3.11 --quiet /content/env_bp
uv pip install --quiet --python /content/env_bp/bin/python basic-pitch==0.4.0 tensorflow==2.15.0 numpy==1.26.4 setuptools==80.9.0 mir_eval pretty_midi soundfile
wget -q -O salamander.tar.xz "https://freepats.zenvoid.org/Piano/SalamanderGrandPiano/SalamanderGrandPiano-SF2-V3+20200602.tar.xz"
tar -xJf salamander.tar.xz && rm salamander.tar.xz
find /content/eval -name "*.sf2"
ls /usr/share/sounds/sf2/
/content/env_demucs/bin/python -c "import torch,demucs;print('demucs ok',torch.cuda.is_available())"
/content/env_bp/bin/python -c "import basic_pitch,mir_eval,pretty_midi;print('bp ok')"
echo SETUP_DONE
''')
open("/content/setup.sh","w").write(script)
subprocess.Popen("nohup bash /content/setup.sh > /content/setup.log 2>&1 &", shell=True)
print("started")

Tesla T4, 15360 MiB

started

Tesla T4, 15360 MiB

started


In [ ]:
BUILD = r'''
"""Build labelled piano mixes: public-domain MIDI, sampled grand piano, a generated band and steam-like noise."""
import json, os, subprocess, urllib.request, glob
import numpy as np, pretty_midi, soundfile as sf
SR = 44100
ROOT = "/content/eval"
SALAMANDER = glob.glob(f"{ROOT}/**/*.sf2", recursive=True)[0]
GM = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
BASE = "https://www.mutopiaproject.org/ftp/"
# Solo piano only, public domain on Mutopia. Alternating per composer, fixed before any result exists.
DEV = ["ChopinFF/O10/10-02-i/10-02-i", "ChopinFF/O10/chp-10-05/chp-10-05", "JoplinS/EliteSyncopations/EliteSyncopations", "JoplinS/SomethingDoing/SomethingDoing", "BeethovenLv/O2/LVB_Sonate_02no1_1/LVB_Sonate_02no1_1", "BeethovenLv/O10/LVB_Sonate_10no1_1/LVB_Sonate_10no1_1", "MozartWA/KV309/KV309_1/KV309_1", "DebussyC/L66/debussy_Arabesque_1/debussy_Arabesque_1", "SchumannR/O15/SchumannOp15No01/SchumannOp15No01", "GriegE/O46/02-lamortdase/02-lamortdase"]
TEST = ["ChopinFF/O10/chopin-op-10-09-wfi/chopin-op-10-09-wfi", "ChopinFF/B130/TroisNouvellesEtudes_Chopin_n1/TroisNouvellesEtudes_Chopin_n1", "JoplinS/PineappleRag/PineappleRag", "JoplinS/WallStreetRag/WallStreetRag", "BeethovenLv/O2/LVB_Sonate_02no1_3/LVB_Sonate_02no1_3", "BeethovenLv/O10/LVB_Sonate_10no1_3/LVB_Sonate_10no1_3", "DebussyC/L75/debussy_Ste_Bergamesq_Clair/debussy_Ste_Bergamesq_Clair", "SchumannR/O15/SchumannOp15No07/SchumannOp15No07", "GriegE/O54/Troldtog/Troldtog", "GriegE/O65/troldhaugen/troldhaugen"]
CLIP = 30.0
os.makedirs(f"{ROOT}/mixes", exist_ok=True); os.makedirs(f"{ROOT}/midi", exist_ok=True)

def render(pm, path, font):
    mid = path.replace(".wav", ".mid"); pm.write(mid)
    subprocess.run(["fluidsynth", "-ni", "-q", "-g", "0.8", "-r", str(SR), "-F", path, font, mid], check=True, capture_output=True)
    x, _ = sf.read(path); x = x.mean(1) if x.ndim > 1 else x
    out = np.zeros(int(CLIP * SR)); n = min(len(x), len(out)); out[:n] = x[:n]; return out

def rms(x): return float(np.sqrt((x ** 2).mean()) + 1e-12)

def steam(rng):
    n = int(CLIP * SR); white = rng.standard_normal(n)
    spec = np.fft.rfft(white); f = np.fft.rfftfreq(n, 1 / SR); spec[f < 3000] = 0; hiss = np.fft.irfft(spec, n)
    env = np.zeros(n)
    for start in (4.0, 13.0, 22.0):
        a, b = int(start * SR), int((start + 4.0) * SR); env[a:b] = 1.0
    return hiss * env

meta = {}
for split, items in (("dev", DEV), ("test", TEST)):
    for rel in items:
        name = rel.split("/")[-1]
        try:
            path = f"{ROOT}/midi/{name}.mid"
            if not os.path.exists(path):
                urllib.request.urlretrieve(BASE + rel + ".mid", path)
            src = pretty_midi.PrettyMIDI(path)
            notes = sorted((n for i in src.instruments if not i.is_drum for n in i.notes), key=lambda n: n.start)
            t0 = notes[0].start
            kept = [n for n in notes if n.start - t0 < CLIP - 0.5]
            piano = pretty_midi.PrettyMIDI(); inst = pretty_midi.Instrument(program=0)
            for n in kept:
                inst.notes.append(pretty_midi.Note(velocity=max(n.velocity, 50), pitch=n.pitch, start=n.start - t0, end=min(n.end - t0, CLIP)))
            piano.instruments.append(inst)
            beats = [b - t0 for b in src.get_beats() if 0 <= b - t0 < CLIP]
            band = pretty_midi.PrettyMIDI(); bass = pretty_midi.Instrument(program=33); drums = pretty_midi.Instrument(program=0, is_drum=True)
            for i, b in enumerate(beats):
                end = beats[i + 1] if i + 1 < len(beats) else b + 0.5
                here = [n.pitch for n in inst.notes if b <= n.start < end]
                if here:
                    p = min(here)
                    while p > 52: p -= 12
                    while p < 31: p += 12
                    bass.notes.append(pretty_midi.Note(velocity=95, pitch=p, start=b, end=max(end - 0.02, b + 0.05)))
                drums.notes.append(pretty_midi.Note(velocity=100, pitch=36 if i % 2 == 0 else 38, start=b, end=b + 0.1))
                drums.notes.append(pretty_midi.Note(velocity=70, pitch=42, start=b, end=b + 0.05))
                drums.notes.append(pretty_midi.Note(velocity=60, pitch=42, start=(b + end) / 2, end=(b + end) / 2 + 0.05))
            band.instruments += [bass, drums]
            a = render(piano, f"{ROOT}/mixes/{name}__piano.wav", SALAMANDER)
            b = render(band, f"{ROOT}/mixes/{name}__backing.wav", GM)
            a = a / rms(a) * 0.1
            backing = b / rms(b) * 0.1 * 10 ** (-2 / 20)
            hiss = steam(np.random.default_rng(abs(hash(name)) % 2**32))
            active = hiss[hiss != 0]
            hiss = hiss / rms(active) * 0.1 * 10 ** (-6 / 20)
            for cond, mix in (("solo", a), ("band", a + backing), ("steam", a + hiss)):
                peak = max(np.abs(mix).max(), 1e-6)
                sf.write(f"{ROOT}/mixes/{name}__{cond}.wav", (mix / peak * 0.9 if peak > 0.9 else mix).astype(np.float32), SR)
            meta[name] = {"split": split, "notes": [[n.start, n.end, n.pitch] for n in inst.notes], "per_second": len(inst.notes) / CLIP}
            print(split, name, len(inst.notes), "notes", flush=True)
        except Exception as error:
            print("SKIP", name, repr(error)[:160], flush=True)
json.dump(meta, open(f"{ROOT}/labels.json", "w"))
print("BUILD_DONE", len(meta), flush=True)
'''
open("/content/build.py","w").write(BUILD)
print(open("/content/setup.log").read()[-600:])

real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

+ mkdir -p /content/eval
+ cd /content/eval
+ uv venv --python 3.11 --quiet /content/env_demucs
+ uv pip install --quiet --python /content/env_demucs/bin/python torch==2.5.1 torchaudio==2.5.1 demucs==4.1.0 numpy==1.26.4 soundfile


real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

+ mkdir -p /content/eval
+ c

In [ ]:
EVAL = r'''
"""Tune Basic Pitch on Demucs piano stems of the dev pieces; report the frozen choice once on test."""
import glob, itertools, json, os
import numpy as np, mir_eval
from basic_pitch import ICASSP_2022_MODEL_PATH
from basic_pitch.constants import AUDIO_SAMPLE_RATE, FFT_HOP
from basic_pitch.inference import predict, Model
from basic_pitch.note_creation import model_output_to_notes
ROOT = "/content/eval"
labels = json.load(open(f"{ROOT}/labels.json"))
DEFAULT = (0.5, 0.3, 30.0, True)   # engine/transcribe.py today
GRID = list(itertools.product((0.3, 0.4, 0.5, 0.6, 0.7), (0.2, 0.3, 0.4), (30.0, 58.0), (True, False)))

def f1(ref, est):
    ref_i = np.array([[a, b] for a, b, _ in ref]); ref_p = mir_eval.util.midi_to_hz(np.array([p for _, _, p in ref], float))
    if not est: return 0.0, 0.0, 0.0
    est_i = np.array([[a, max(b, a + 0.01)] for a, b, _ in est]); est_p = mir_eval.util.midi_to_hz(np.array([p for _, _, p in est], float))
    p, r, f, _ = mir_eval.transcription.precision_recall_f1_overlap(ref_i, ref_p, est_i, est_p, onset_tolerance=0.05, offset_ratio=None)
    return float(f), float(p), float(r)

# The harness has to behave before any model is judged.
ref = [[0.5, 1.0, 60], [1.0, 1.5, 64], [1.5, 2.0, 67]]
assert f1(ref, ref)[0] == 1.0
assert f1(ref, [[a + 0.03, b, p] for a, b, p in ref])[0] == 1.0
assert f1(ref, [[a + 0.08, b, p] for a, b, p in ref])[0] == 0.0
assert f1(ref, [[a, b, p + 12] for a, b, p in ref])[0] == 0.0
assert abs(f1(ref, ref[:2])[0] - 0.8) < 1e-9
print("harness ok", flush=True)

def source(name, cond):
    return f"{ROOT}/mixes/{name}__solo.wav" if cond == "solo" else f"{ROOT}/stems/htdemucs_6s/{name}__{cond}/piano.wav"

model = Model(ICASSP_2022_MODEL_PATH)
outputs = {}
for name in labels:
    for cond in ("solo", "band", "steam"):
        path = source(name, cond)
        if os.path.exists(path):
            outputs[(name, cond)] = predict(path, model)[0]
    print("predicted", name, flush=True)

def notes(output, cfg):
    onset, frame, length, melodia = cfg
    _, events = model_output_to_notes(output, onset_thresh=onset, frame_thresh=frame, min_note_len=int(np.round(length / 1000 * (AUDIO_SAMPLE_RATE / FFT_HOP))), include_pitch_bends=False, melodia_trick=melodia)
    return [[float(e[0]), float(e[1]), int(e[2])] for e in events]

def score(cfg, split, conds):
    rows = [f1(labels[n]["notes"], notes(o, cfg)) + (n, c) for (n, c), o in outputs.items() if labels[n]["split"] == split and c in conds]
    return rows

def mean(rows, i=0): return round(float(np.mean([r[i] for r in rows])), 4)

table = []
for cfg in GRID:
    rows = score(cfg, "dev", ("band", "steam"))
    table.append((mean(rows), mean(rows, 1), mean(rows, 2), cfg))
table.sort(key=lambda t: -t[0])
print("DEV top 8 (F1, precision, recall, onset, frame, min ms, melodia)", flush=True)
for row in table[:8]: print(row, flush=True)
default_dev = [t for t in table if t[3] == DEFAULT][0]
print("DEV default", default_dev, flush=True)
chosen = table[0][3]
print("CHOSEN on dev, frozen:", chosen, flush=True)

report = {"chosen": chosen, "dev_default": default_dev[:3], "dev_chosen": table[0][:3]}
for cond in ("band", "steam", "solo"):
    d = score(DEFAULT, "test", (cond,)); c = score(chosen, "test", (cond,))
    report[cond] = {"default": [mean(d), mean(d, 1), mean(d, 2)], "chosen": [mean(c), mean(c, 1), mean(c, 2)], "n": len(d),
                    "per_piece_gain": sorted(round(x[0] - y[0], 3) for x, y in zip(c, d))}
print("TEST", json.dumps(report), flush=True)
json.dump(report, open(f"{ROOT}/report.json", "w"))
print("EVAL_DONE", flush=True)
'''
open("/content/eval.py","w").write(EVAL)
RUN = r'''
set -x
until grep -q SETUP_DONE /content/setup.log; do sleep 5; done
cd /content/eval
/content/env_bp/bin/python /content/build.py
mkdir -p stems
FILES=$(ls mixes/*__band.wav mixes/*__steam.wav)
/content/env_demucs/bin/python -m demucs -n htdemucs_6s -d cuda --out stems $FILES
echo DEMUCS_DONE
/content/env_bp/bin/python /content/eval.py
echo ALL_DONE
'''
open("/content/run.sh","w").write(RUN)
import subprocess
subprocess.Popen("nohup bash /content/run.sh > /content/run.log 2>&1 &", shell=True)
print(open("/content/setup.log").read()[-500:])

so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

+ mkdir -p /content/eval
+ cd /content/eval
+ uv venv --python 3.11 --quiet /content/env_demucs
+ uv pip install --quiet --python /content/env_demucs/bin/python torch==2.5.1 torchaudio==2.5.1 demucs==4.1.0 numpy==1.26.4 soundfile


so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

+ mkdir -p /content/eval
+ cd /content/eval
+ uv venv --python 3.11 --quiet /content/env_demucs
+ uv pip install --quiet --python /content/env_demucs/bin/python torch==2.5.1 torchaudio==2.5.1 demucs==4.1.0 numpy==1.26.4 soundfil

In [ ]:
import subprocess
print("SETUP:", subprocess.run("tail -c 500 /content/setup.log", shell=True, capture_output=True, text=True).stdout)
print("RUN:", subprocess.run("grep -vE 'it/s|s/it|%\\|' /content/run.log | tail -n 25 | cut -c1-400", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("ls /content/eval/mixes 2>/dev/null | wc -l; ls /content/eval/stems/htdemucs_6s 2>/dev/null | wc -l", shell=True, capture_output=True, text=True).stdout)

SETUP: PU will not be used.
2026-11-02 12:02:50.821906: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-02 12:02:54.442240: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
bp ok
+ echo SETUP_DONE
SETUP_DONE

RUN: Separating track mixes/debussy_Arabesque_1__steam.wav
Separating track mixes/debussy_Ste_Bergamesq_Clair__band.wav
Separating track mixes/debussy_Ste_Bergamesq_Clair__steam.wav
Separating track mixes/EliteSyncopations__band.wav
Separating track mixes/EliteSyncopations__steam.wav
Separating track mixes/KV309_1__band.wav
Separating track mixes/KV309_1__steam.wav
Separating track mixes/LVB_Sonate_02no1_1__band.wav
Separating track mixes/LVB_Sonate_02no1_1__steam.wav
Separating track mi

In [ ]:
import json
r = json.load(open("/content/eval/report.json"))
print(json.dumps(r, indent=0)[:1800])
labels = json.load(open("/content/eval/labels.json"))
print({k: sum(1 for v in labels.values() if v["split"] == k) for k in ("dev", "test")}, sum(len(v["notes"]) for v in labels.values()), "notes")

{
"chosen": [
0.6,
0.3,
30.0,
false
],
"dev_default": [
0.716,
0.662,
0.8026
],
"dev_chosen": [
0.7686,
0.8012,
0.7571
],
"band": {
"default": [
0.683,
0.5798,
0.8699
],
"chosen": [
0.7514,
0.7051,
0.8328
],
"n": 10,
"per_piece_gain": [
0.005,
0.007,
0.025,
0.05,
0.063,
0.073,
0.088,
0.1,
0.109,
0.164
]
},
"steam": {
"default": [
0.7527,
0.6824,
0.8643
],
"chosen": [
0.8239,
0.8227,
0.8373
],
"n": 10,
"per_piece_gain": [
-0.009,
0.032,
0.042,
0.043,
0.044,
0.049,
0.051,
0.082,
0.134,
0.243
]
},
"solo": {
"default": [
0.7908,
0.6738,
0.9715
],
"chosen": [
0.8849,
0.8205,
0.9625
],
"n": 10,
"per_piece_gain": [
0.022,
0.063,
0.063,
0.067,
0.071,
0.079,
0.095,
0.107,
0.127,
0.247
]
}
}
{'dev': 10, 'test': 10} 5221 notes
